# 06 Evaluation and Optimisation (Phase 6)

**Synthetic RAW 50MP Lab.** This notebook closes the experiment's first cycle (PRD Phase 6: full ablations,
controlled reference test, visual review, performance measurements, final report; exit criterion: results
sufficient to accept or reject the experiment's hypotheses):

- **Run collection:** the run folders or zips of notebooks 00 to 03 (and 05) are read, and their manifests
  compared by source image: source consistency of every output, seam gates, identity and night metrics,
  runtime and memory. The captain's notebook 00 Colab runs ship with the repository (`evidence/colab_runs`);
  add the notebook 01, 02 and 03 zips with `RUN_SOURCES` or `UPLOAD_RUN_ZIPS`.
- **Visual review sheet:** identical crops from every run of one source, at 100% from the masters when they can
  be found (in place or under `MASTERS_ROOTS`, for example a Drive copy), otherwise from the review previews.
- **Full-pipeline ablation (E09, PRD 31 matrix, E02, E07):** one input processed with each stage switched off
  in turn (SR mode, specialised processing, face handling, grain, optics, sensor simulation), with metrics,
  runtime and memory per configuration, the PRD 28.3 SR settings sweep and the E01 bit-depth study.
- **Controlled reference test (E10):** real, freely licensed high-resolution photographs (fetched at run time
  from Wikimedia Commons, licence recorded) degraded to the source scale and reconstructed, the only place
  accuracy against ground truth can be measured.
- **Final experiment report:** every PRD 34 question answered with evidence or an explicit "not measured", a
  PRD 33 outcome recommendation and the best quality-to-runtime Colab configuration.

All processing code lives in the shared `synthetic_raw_lab` package (`src/evaluation_config.py`,
`src/ablation.py`, `src/reference_test.py`, `src/run_collection.py`, `src/final_report.py`); this notebook only
sets parameters, calls the package and displays results.

## 1. Introduction and scientific limitations

- The two inputs are AI renders with **no ground truth**: on them, only source consistency (does the output
  reduce back to the source?) and change measurements can be made. Neither proves added detail is correct.
- The **reference test** supplies ground truth by degrading real photographs. Its degradation models (ideal
  sampling, a camera forward model, an AI-render proxy) are assumptions about how a render like the inputs
  could have been formed; results transfer to the inputs only as far as those assumptions hold.
- **Above-limit error** (`hf NMSE`) is the error in the band finer than the source sampling limit, relative to
  that band's energy in the truth: about 1.0 is what adding no detail scores, below 1 means real detail was
  recovered, above 1 means the added detail is wrong more than right.
- The ablation processes a **crop** at the scale a full run uses, so runtimes are scaled to full size as
  estimates; the notebook 00 to 03 runs give the measured full-size runtimes.
- Every threshold is provisional, to be calibrated experimentally (PRD 25.3). Outputs are synthetic
  camera-grade masters, never camera RAW.

## 2. Runtime setup

Set the run parameters. In Google Colab these are form fields. Use a **GPU runtime** (Runtime, Change runtime
type) for the ablation and the reference test: they run pretrained SR.

- **Repository and code version:** the setup cell clones `REPO_URL` at `REPO_REF` into `/content/experiment_50MP`
  with no token; `PROJECT_DIR` uses an existing checkout instead.
- **Keep results:** set `MOUNT_DRIVE` and point `OUTPUT_ROOT`, `REPORTS_ROOT` and `INPUTS_ROOT` at Drive.
- **Runs to compare:** `INCLUDE_REPOSITORY_EVIDENCE` adds the notebook 00 runs in `evidence/colab_runs`;
  `RUN_SOURCES` lists run folders, folders of runs or report zips (for example on Drive); `UPLOAD_RUN_ZIPS`
  asks for zips in Colab. `MASTERS_ROOTS` lists folders where full-resolution masters can be found by name.
- **Ablation:** `RUN_ABLATION` runs it on `ABLATION_INPUT` (`upload`, `path` with `ABLATION_INPUT_PATH`, or a
  synthetic scene); `ABLATION_SCENE` overrides the automatic scene kind; `ABLATION_CROP_SIZE` is the source
  crop side (0 processes the whole input at the full target, as slow as notebook 00).
- **Reference test:** `RUN_REFERENCE_TEST` downloads the `REFERENCES` (about 13 and 20 MB) into the input store
  and reconstructs `REFERENCE_CROP_MEGAPIXELS` of each with every degradation model.
- **Blind review:** score the blind sheets first (section 12); `REVEAL_BLIND_KEY` then shows the key, the
  labelled review sheets and the ablation tables, and `REVIEW_SCORES` holds your PRD 26.3 scores per
  configuration. After setting either, re-run only this cell and sections 12 and 13: re-running sections 6 to
  10 would collect the runs again and repeat the ablation and the reference test.
- **Advanced:** `ADVANCED` updates sections of the evaluation profile, for example
  `{"ablation": {"configurations": ["lanczos", "sr_conservative", "full"]}}` or
  `{"reference": {"scales": [1.7134, 2.4495]}}`. Every key is listed in section 5.

In [ ]:
# @title Run parameters
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
PROJECT_DIR = ""  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
OUTPUT_ROOT = ""  # @param {type:"string"}
REPORTS_ROOT = ""  # @param {type:"string"}
INPUTS_ROOT = ""  # @param {type:"string"}
CONFIG_FILE = "configs/evaluation_default.yaml"  # @param {type:"string"}

# @markdown Runs to compare (notebooks 00 to 05)
INCLUDE_REPOSITORY_EVIDENCE = True  # @param {type:"boolean"}
RUN_SOURCES = []  # @param {type:"raw"}
UPLOAD_RUN_ZIPS = False  # @param {type:"boolean"}
MASTERS_ROOTS = []  # @param {type:"raw"}

# @markdown Full-pipeline ablation (E09)
RUN_ABLATION = True  # @param {type:"boolean"}
ABLATION_INPUT = "upload"  # @param ["upload", "path", "synthetic_night", "synthetic_crowd"]
ABLATION_INPUT_PATH = ""  # @param {type:"string"}
ABLATION_SCENE = "auto"  # @param ["auto", "human", "night", "general"]
ABLATION_CROP_SIZE = 1536  # @param {type:"integer"}
SYNTHETIC_SIZE = "1024x768"  # @param {type:"string"}

# @markdown Controlled reference test (E10)
RUN_REFERENCE_TEST = True  # @param {type:"boolean"}
REFERENCES = "portrait_holi,night_forest_path"  # @param {type:"string"}
REFERENCE_CROP_MEGAPIXELS = 12.0  # @param {type:"number"}

# @markdown Super-resolution (Phase 2 backend)
ENABLE_SUPER_RESOLUTION = True  # @param {type:"boolean"}
SR_BACKEND = "swinir"  # @param ["swinir", "realesrgan"]
WEIGHTS_DIR = ""  # @param {type:"string"}

# @markdown Blind review scores (PRD 26.3) and advanced settings
REVEAL_BLIND_KEY = False  # @param {type:"boolean"}
REVIEW_SCORES = {}  # @param {type:"raw"}
ADVANCED = {}  # @param {type:"raw"}

## 3. Dependency installation

Finds the repository (`PROJECT_DIR`, the checkout this notebook lives in, or the Colab clone of `REPO_REF`) and
installs the `synthetic_raw_lab` package from it (plus the `sr` extra, PyTorch and spandrel, when
super-resolution is enabled), like notebooks 00 to 05. The exact package versions used are recorded in every
manifest.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

NOTEBOOK_VERSION = "06_Evaluation/1.0"
COLAB_CLONE = Path("/content/experiment_50MP")
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project() -> Path:
    if PROJECT_DIR:
        return Path(PROJECT_DIR).expanduser().resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pipeline.py").exists():
            return candidate.resolve()
    return COLAB_CLONE


def git(*args: str) -> None:
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"git failed (check REPO_URL and REPO_REF): {result.stderr}")


PROJECT = find_project()
if not (PROJECT / "pyproject.toml").exists():
    git("clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT))
elif PROJECT == COLAB_CLONE:
    git("-C", str(PROJECT), "fetch", "--depth", "1", REPO_URL, REPO_REF)
    git("-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD")
spec = importlib.util.find_spec("synthetic_raw_lab")
missing_sr = ENABLE_SUPER_RESOLUTION and importlib.util.find_spec("spandrel") is None
if spec is None or Path(spec.origin).resolve().parent != PROJECT / "src" or missing_sr:
    target = f"{PROJECT}[sr]" if ENABLE_SUPER_RESOLUTION else str(PROJECT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], check=True)
    importlib.invalidate_caches()
for name in [m for m in sys.modules if m.partition(".")[0] == "synthetic_raw_lab"]:
    del sys.modules[name]

import synthetic_raw_lab  # noqa: E402
from synthetic_raw_lab import environment  # noqa: E402

print(f"project: {PROJECT} (Git: {environment.git_revision(PROJECT)})")
print(f"synthetic_raw_lab {synthetic_raw_lab.__version__} from {Path(synthetic_raw_lab.__file__).parent}")

## 4. GPU and environment inspection

Loads the evaluation profile (`CONFIG_FILE`, then the form fields and `ADVANCED`) so every later cell uses the
same settings, then reports GPU, CUDA driver, PyTorch, the SR device, RAM and free disk (PRD 21.2).

In [ ]:
import json

from IPython.display import Image, Markdown, display

from synthetic_raw_lab import ablation, evaluation_config, final_report, pipeline, reference_test, sr_models
from synthetic_raw_lab import run_collection as runs_lib


def table(header, rows):
    return Markdown(runs_lib.markdown_table(header, rows))


universal = {"super_resolution": {"enabled": ENABLE_SUPER_RESOLUTION, "backend": SR_BACKEND}}
if WEIGHTS_DIR:
    universal["super_resolution"]["weights_dir"] = WEIGHTS_DIR
universal.update(
    {
        k: v
        for k, v in {"output_root": OUTPUT_ROOT, "reports_root": REPORTS_ROOT, "inputs_root": INPUTS_ROOT}.items()
        if v
    }
)
sections = {
    "ablation": {"scene": ABLATION_SCENE, "crop_size": int(ABLATION_CROP_SIZE)},
    "reference": {
        "references": [r.strip() for r in REFERENCES.split(",") if r.strip()],
        "crop_megapixels": float(REFERENCE_CROP_MEGAPIXELS),
    },
}
for key, value in dict(ADVANCED).items():
    if key == "universal_overrides":
        for name, override in value.items():
            merge = isinstance(override, dict) and isinstance(universal.get(name), dict)
            universal[name] = {**universal[name], **override} if merge else override
    elif isinstance(value, dict):
        sections[key] = {**sections.get(key, {}), **value}
    else:
        sections[key] = value
config_path = Path(CONFIG_FILE) if Path(CONFIG_FILE).is_absolute() else PROJECT / CONFIG_FILE
CONFIG = evaluation_config.load_evaluation_config(config_path, PROJECT, universal_overrides=universal, **sections)
EXPERIMENT = CONFIG.experiment
INPUTS_DIR, OUTPUT_DIR, REPORTS_DIR = pipeline.resolve_roots(EXPERIMENT, PROJECT)

runtime = environment.inspect_runtime(OUTPUT_DIR)
gpu = runtime["gpu"]
print(f"Python {runtime['python']} on {runtime['platform']} (Colab: {runtime['in_colab']})")
print(
    f"GPU: {gpu['name']} {gpu['memory_total_gb']} GB, driver CUDA {gpu['driver_cuda_version']}"
    if gpu
    else "GPU: none detected"
)
print(f"PyTorch: {runtime['packages'].get('torch', 'not installed')}")
if not CONFIG.universal.sr_enabled:
    print("super-resolution: disabled (SR configurations are skipped and say so)")
elif sr_models.available():
    device = sr_models.device_info(sr_models.resolve_device(CONFIG.universal.super_resolution.device))
    print(f"super-resolution: {SR_BACKEND} on {device['name']} ({device['device']})")
else:
    print(f"WARNING: SR configurations will be skipped: {pipeline.SR_INSTALL_HINT}")
print(f"RAM: {runtime['ram_available_gb']} of {runtime['ram_total_gb']} GB available")
print(f"free disk: {runtime['disk_free_gb']} GB")

## 5. Evaluation configuration

Every setting of the evaluation profile (`configs/evaluation_default.yaml`, documented in
`src/evaluation_config.py`), then the ablation configurations with the stages each one runs.

In [ ]:
display(
    table(
        ["parameter", "value", "default"],
        [
            (f"`{r['parameter']}`", f"`{r['current']}`", f"`{r['default']}`")
            for r in evaluation_config.parameter_rows(CONFIG)
        ],
    )
)
display(
    table(
        ["configuration", "stages", "purpose"],
        [
            (f"`{c['name']}`", ", ".join(c["stages"]), c["purpose"])
            for c in ablation.configuration_rows(CONFIG.ablation.configurations)
        ],
    )
)

## 6. Run collection

Collects every run folder, folder of runs and zip: the repository's notebook 00 runs, `RUN_SOURCES`, uploaded
zips and, after sections 9 and 10, this notebook's own ablation and reference-test runs. Each manifest is
recognised by its schema; a notebook 03 run's delegated notebook 01 or 02 run is linked to it.

In [ ]:
WORK_DIR = REPORTS_DIR / "_collected"
SOURCES = [Path(p).expanduser() for p in RUN_SOURCES]
if INCLUDE_REPOSITORY_EVIDENCE:
    SOURCES.append(PROJECT / "evidence" / "colab_runs")
if UPLOAD_RUN_ZIPS:
    if not IN_COLAB:
        raise RuntimeError("UPLOAD_RUN_ZIPS only works in Colab; list the zips in RUN_SOURCES instead")
    from google.colab import files

    UPLOADS = WORK_DIR / "_uploads"
    UPLOADS.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (UPLOADS / name).write_bytes(data)
        SOURCES.append(UPLOADS / name)


def collect():
    return runs_lib.collect(SOURCES, WORK_DIR, CONFIG.aggregation)


RUNS = collect()
display(table(*runs_lib.inventory_table(RUNS)))
missing = {"01", "02", "03"} - {r.notebook for r in RUNS}
if missing:
    print(f"not collected yet: notebooks {', '.join(sorted(missing))} (add their zips to RUN_SOURCES)")

## 7. Run comparison

Source consistency of every output after area reduction to the source size (PRD 23.2), then performance
(PRD 28.1), night metrics before and after the Layer 5B processing (PRD 23.5) and identity by face tier
(PRD 11.3), for the runs that measured them.

In [ ]:
for title, (header, rows) in (
    ("Source consistency", runs_lib.outputs_table(RUNS)),
    ("Performance", runs_lib.performance_table(RUNS)),
    ("Night metrics", runs_lib.night_table(RUNS)),
    ("Identity by face tier", runs_lib.identity_table(RUNS)),
):
    display(Markdown(f"**{title}**"))
    display(table(header, rows) if rows else Markdown("no collected run measured this yet"))

## 8. Visual review sheet

Identical crops from every run of each source (PRD 26): rows are run outputs, columns are crops (the crops and
faces the runs recorded, or `aggregation.review_crops`). A row cut from a full-resolution master shows 100%
output pixels; a row marked `(preview)` comes from the run's review preview because its master was not found
(add the masters folder to `MASTERS_ROOTS`). Review at 100% and 200% for halos, ringing, seams, invented texture
and face changes (PRD 25.2).

In [ ]:
SHEETS = {}
for sha, group in runs_lib.by_source(RUNS).items():
    group = [r for r in group if r.notebook in ("00", "01", "02", "03")]
    if not group:
        continue
    try:
        sheet = runs_lib.review_sheet(
            group, REPORTS_DIR / "_review" / f"review_{sha[:12]}.png", CONFIG.aggregation, MASTERS_ROOTS
        )
    except ValueError as error:
        print(f"{group[0].label}: no review sheet ({error})")
        continue
    SHEETS[group[0].label] = sheet["path"]
    print(f"{group[0].label}: " + ", ".join(f"{row['row']} ({row.get('scale', 'source')})" for row in sheet["rows"]))
    display(Image(filename=sheet["path"]))

## 9. Full-pipeline ablation

Runs the PRD 31 matrix and the E09 ablation on one input (`ABLATION_INPUT`): every configuration of section 5 on
the same crop, processed at the scale a full 50 MP run of the whole input uses. Each upscale runs once and is
shared; each configuration records its stages, runtime (with the shared upscale counted), peak memory, detail
energy, source consistency and clipping. Then the stage contributions (full pipeline minus one stage), the SR
settings sweep (PRD 28.3) and the E01 bit-depth study. Run it once per input (portrait, then night).

This section shows only the **blind sheet** (configurations under letters, in shuffled order; PRD 26.3): the
labelled review sheet, the key and the configuration tables stay hidden until you have scored it (section 12).

In [ ]:
from PIL import Image as PILImage

from synthetic_raw_lab import night_processing as night
from synthetic_raw_lab import test_images

ABLATION = None
if RUN_ABLATION:
    staging = OUTPUT_DIR / "_staging"
    staging.mkdir(parents=True, exist_ok=True)
    if ABLATION_INPUT == "upload":
        if not IN_COLAB:
            raise RuntimeError("upload only works in Colab; use ABLATION_INPUT='path' or a synthetic scene")
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError(f"upload exactly one image, got {len(uploaded)}")
        name, data = next(iter(uploaded.items()))
        ABLATION_SOURCE = staging / name
        ABLATION_SOURCE.write_bytes(data)
    elif ABLATION_INPUT == "path":
        ABLATION_SOURCE = Path(ABLATION_INPUT_PATH).expanduser()
    else:
        width, height = (int(v) for v in SYNTHETIC_SIZE.lower().split("x"))
        ABLATION_SOURCE = staging / f"{ABLATION_INPUT}_{width}x{height}.png"
        if ABLATION_INPUT == "synthetic_night":
            night.write_night_scene(ABLATION_SOURCE, width, height)
        else:
            pixels, faces = test_images.make_crowd_test_image(width, height)
            PILImage.fromarray(pixels).save(ABLATION_SOURCE)
            boxes = [[f.x, f.y, f.width, f.height] for f in faces]
            universal["analysis"] = {
                **universal.get("analysis", {}),
                "face_detection": False,
                "manual_face_boxes": boxes,
            }
            CONFIG = evaluation_config.load_evaluation_config(
                config_path, PROJECT, universal_overrides=universal, **sections
            )
    try:
        ABLATION = ablation.run_ablation(ABLATION_SOURCE, CONFIG, PROJECT, notebook_version=NOTEBOOK_VERSION)
    except pipeline.StageError as error:
        print(f"FAILED in stage {error.stage}: {error}")
        print(
            "Reduced-memory retry: a smaller ABLATION_CROP_SIZE, or ADVANCED "
            "{'universal_overrides': {'super_resolution': {'tile_size': 256}}}."
        )
        raise
    manifest = ABLATION.manifest
    print(f"ablation {ABLATION.run_id}: {manifest['status']} in {manifest['total_seconds']:.0f} s")
    print(f"scene {manifest['scene']['kind']}, crop {manifest['crop']['box']} -> {manifest['crop']['output_size']}")
    display(Markdown("**Blind sheet** (score it in section 12 before revealing the key)"))
    display(Image(filename=manifest["figures"]["blind_sheet"]))
    SOURCES.append(ABLATION.reports_dir)
else:
    print("RUN_ABLATION is off")

## 10. Controlled reference test

Experiment E10. Each reference photograph is fetched from Wikimedia Commons (licence, author and page read from
the Commons API and checked against the free licences allowed; the file verified against its pinned SHA-1),
centre-cropped, degraded to the source scale by each model and reconstructed by the reference configurations.
Accuracy is measured against the original: PSNR, SSIM, the low band the source could hold, and the above-limit
error and correlation that say how much added detail is real. The reconstructions are adaptations of the
photographs under their licences; keep the attribution with them.

In [ ]:
REFERENCE = None
if RUN_REFERENCE_TEST:
    try:
        REFERENCE = reference_test.run_reference_test(CONFIG, PROJECT, notebook_version=NOTEBOOK_VERSION)
    except pipeline.StageError as error:
        print(f"FAILED in stage {error.stage}: {error}")
        raise
    seconds = REFERENCE.manifest["total_seconds"]
    print(f"reference test {REFERENCE.run_id}: {REFERENCE.manifest['status']} in {seconds:.0f} s")
    display(Markdown(REFERENCE.report_path.read_text()))
    SOURCES.append(REFERENCE.reports_dir)
else:
    print("RUN_REFERENCE_TEST is off")

## 11. Performance measurements

Measured runtimes of the collected runs (PRD 28.1) next to the ablation's per-configuration runtimes scaled to
the full 50 MP size, and the SR settings sweep (tile size, overlap, precision; PRD 28.3).

In [ ]:
RUNS = collect()
evidence = final_report.Evidence(RUNS)
display(table(*runs_lib.performance_table(RUNS)))
estimates = final_report.full_size_runtime(evidence)
if estimates:
    display(
        table(
            ["configuration", "estimated full-size seconds", "GPU"],
            [(name, f"{v['estimated_full_seconds']:.0f}", v["gpu"]) for name, v in estimates.items()],
        )
    )
for run in evidence.ablations:
    sweep = run.manifest.get("performance_sweep") or []
    if sweep:
        display(Markdown(f"**SR settings sweep, {run.label}**"))
        display(
            table(
                [
                    "setting",
                    "SR s (no model load)",
                    "tile",
                    "precision",
                    "peak GPU MB",
                    "seam gate",
                    "PSNR vs default dB",
                    "note",
                ],
                [
                    (
                        s.get("setting"),
                        runs_lib.fmt(s.get("seconds"), 1),
                        runs_lib.fmt(s.get("tile_size")),
                        runs_lib.fmt(s.get("precision")),
                        runs_lib.fmt(s.get("peak_device_mb"), 0),
                        runs_lib.fmt(s.get("seam_gate_passed")) if s.get("seam_gate_passed") is not None else "-",
                        runs_lib.fmt(s.get("psnr_vs_default_db")),
                        s.get("skipped") or s.get("error") or "",
                    )
                    for s in sweep
                ],
            )
        )

## 12. Blind review

PRD 26.3: score every letter of each ablation's **blind sheet** (configurations under letters, in shuffled order)
from 1 to 5 for naturalness, structural fidelity, perceived detail, artefact severity (higher is worse) and
overall preference. Sections 9 and 12 show only the blind sheets: the key, the labelled review sheets and the
ablation tables stay hidden until you have scored them. Then set `REVEAL_BLIND_KEY` in section 2, run that cell
and the second cell below, translate your letters with the key and enter the scores in `REVIEW_SCORES`, for
example
`{"lanczos": {"naturalness": 3, "overall_preference": 2}, "sr_conservative": {"naturalness": 4, "overall_preference": 4}}`,
then re-run only the section 2 parameter cell and sections 12 and 13 (not the ablation or the reference test,
which would run again). Outcome A in the final report needs a blind preference for SR over Lanczos.

In [ ]:
for run in evidence.ablations:
    blind = (run.manifest.get("figures") or {}).get("blind_sheet")
    if blind and Path(blind).exists():
        display(Markdown(f"**Blind sheet, {run.label}**"))
        display(Image(filename=blind))
print("scores entered:", json.dumps(REVIEW_SCORES) if REVIEW_SCORES else "none yet")

In [ ]:
if REVEAL_BLIND_KEY or REVIEW_SCORES:
    for run in evidence.ablations:
        figures = run.manifest.get("figures") or {}
        key = figures.get("blind_key") or {}
        display(Markdown(f"**Key, {run.label}:** " + ", ".join(f"{letter} = `{name}`" for letter, name in key.items())))
        sheet = figures.get("review_sheet")
        if sheet and Path(sheet).exists():
            display(Image(filename=sheet))
        if run.has_report:
            report = (run.folder / "report.md").read_text()
            start = report.find("## 3.")
            if start >= 0:
                display(Markdown(report[start:]))
else:
    print(
        "The key, the labelled review sheets and the ablation tables stay hidden until the blind sheets are "
        "scored: then set REVEAL_BLIND_KEY in section 2, run that cell and run this one."
    )

## 13. Final experiment report

Builds the final report from every collected run: each PRD 34 question with its status (answered, partial, not
measured, reasoned) and evidence, the PRD 33 outcome recommendation, the best quality-to-runtime Colab
configuration and what this upscaling approach can and cannot achieve. Collecting more runs and re-running this
section updates every answer.

In [ ]:
import shutil

RECORD = final_report.build(RUNS, CONFIG, REVIEW_SCORES or None)
FINAL_DIR = REPORTS_DIR / f"final_report_{RECORD['generated_at'][:19].replace(':', '').replace('-', '')}"
FIGURES = {}
for name, path in SHEETS.items():
    target = FINAL_DIR / "figures" / Path(path).name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(path, target)
    FIGURES[name] = f"figures/{target.name}"
PATHS = final_report.write(RECORD, RUNS, FINAL_DIR, FIGURES)
display(Markdown(PATHS["report"].read_text()))
print(f"report: {PATHS['report']}")

## 14. Download and archive

Zips the final report, this notebook's ablation and reference-test run folders and the review sheets
(no full-resolution masters; PRD 21.5) and downloads the zip in Colab.

In [ ]:
import zipfile

ARCHIVE = FINAL_DIR.with_suffix(".zip")
folders = [FINAL_DIR, REPORTS_DIR / "_review"]
folders += [result.reports_dir for result in (ABLATION, REFERENCE) if result is not None]
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in folders:
        if folder.exists():
            for path in sorted(folder.rglob("*")):
                if path.is_file():
                    bundle.write(path, path.relative_to(folder.parent))
print(f"archive: {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(str(ARCHIVE))